# 02 — Data Preprocessing

## Marketing Intelligence for Campaign Optimization

### Objective

The objective of this notebook is to prepare the advertising campaign dataset for exploratory analysis and predictive modeling.

The preprocessing stage includes:

- Loading the raw dataset
- Creating a working copy
- Inspecting and correcting data types
- Converting date variables
- Handling zero-conversion cases
- Preparing categorical and ordinal variables
- Identifying derived and leakage-prone variables
- Validating the processed dataset
- Saving the cleaned dataset

The original raw dataset will not be modified.

In [20]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [23]:
df = pd.read_csv("../data/raw/tech_advertising_campaigns_dataset.csv")
print("Dataset loaded successfully.")

Dataset loaded successfully.


In [24]:
df_clean = df.copy()

print("Working copy created.")
print("Original shape:", df.shape)
print("Working copy shape:", df_clean.shape)

Working copy created.
Original shape: (10000, 41)
Working copy shape: (10000, 41)


In [25]:
df_clean.dtypes

campaign_id                         str
campaign_objective                  str
platform                            str
ad_placement                        str
device_type                         str
operating_system                    str
creative_format                     str
creative_size                       str
ad_copy_length                      str
has_call_to_action                 bool
creative_emotion                    str
creative_age_days                 int64
target_audience_age                 str
target_audience_gender              str
audience_interest_category          str
income_bracket                      str
purchase_intent_score               str
retargeting_flag                   bool
start_date                          str
quarter                           int64
day_of_week                         str
hour_of_day                       int64
campaign_day                      int64
quality_score                     int64
actual_cpc                      float64


In [26]:
# Remove redundant duplicate column
df_clean = df_clean.drop(columns=["actual_cpc"])

print("Updated dataset shape:", df_clean.shape)
print("\nactual_cpc removed successfully.")

Updated dataset shape: (10000, 40)

actual_cpc removed successfully.


In [27]:
df_clean["start_date"] = pd.to_datetime(
    df["start_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

In [28]:
print("Data type:", df_clean["start_date"].dtype)
print("Invalid/missing dates:", df_clean["start_date"].isna().sum())
print("Earliest date:", df_clean["start_date"].min())
print("Latest date:", df_clean["start_date"].max())

Data type: datetime64[us]
Invalid/missing dates: 0
Earliest date: 2024-01-01 00:00:00
Latest date: 2026-01-30 00:00:00


In [29]:
print("Unique dates:", df_clean["start_date"].nunique())
print("Total records:", len(df_clean))

print("\nSample converted dates:")
print(df_clean["start_date"].head(10))

Unique dates: 761
Total records: 10000

Sample converted dates:
0   2024-03-06
1   2024-01-26
2   2025-05-15
3   2024-07-21
4   2025-03-09
5   2024-10-29
6   2024-08-22
7   2025-05-01
8   2025-12-27
9   2024-09-17
Name: start_date, dtype: datetime64[us]


In [30]:
print("\nDate frequency:")
print(
    df_clean["start_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
    .head(12)
)


Date frequency:
start_date
2024-01    409
2024-02    409
2024-03    411
2024-04    407
2024-05    439
2024-06    373
2024-07    395
2024-08    426
2024-09    383
2024-10    392
2024-11    343
2024-12    434
Freq: M, Name: count, dtype: int64


In [31]:
zero_conversions = (df_clean["conversions"] == 0).sum()

print("Campaigns with zero conversions:", zero_conversions)

print(
    "Percentage with zero conversions:",
    round((zero_conversions / len(df_clean)) * 100, 2),
    "%"
)

Campaigns with zero conversions: 594
Percentage with zero conversions: 5.94 %


In [32]:
zero_conversion_cpa = df_clean.loc[
    df_clean["conversions"] == 0,
    "CPA"
]

print(zero_conversion_cpa.describe())
print("\nUnique CPA values for zero-conversion campaigns:")
print(zero_conversion_cpa.unique()[:20])

count    594.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: CPA, dtype: float64

Unique CPA values for zero-conversion campaigns:
[0.]


## Zero-Conversion and CPA Handling

The dataset contains 594 campaigns with zero conversions, representing 5.94% of all observations.

For these campaigns, the dataset records CPA as 0. Mathematically, CPA is undefined when conversions are zero because CPA is calculated as:

CPA = Ad Spend / Conversions

Therefore, a stored CPA value of 0 for zero-conversion campaigns should not be interpreted as a true acquisition cost of zero.

The original CPA values will be retained in the cleaned dataset rather than replacing them with arbitrary values, missing values, or infinite values. During exploratory analysis, zero-conversion campaigns will be interpreted separately when analyzing CPA.

No rows will be removed because of zero conversions, since a campaign receiving clicks but producing no conversions is a valid marketing outcome and is important for campaign optimization.

In [36]:
derived_variables = {
    "CTR": "clicks / impressions × 100",
    "conversion_rate": "conversions / clicks × 100",
    "ROAS": "revenue / ad_spend",
    "profit": "revenue - ad_spend",
    "CPC": "ad_spend / clicks",
    "CPA": "ad_spend / conversions"
}

derived_variable_summary = pd.DataFrame(
    list(derived_variables.items()),
    columns=["Variable", "Derived_From"]
)

derived_variable_summary

,Variable,Derived_From
0,CTR,clicks / impressions × 100
1,conversion_rate,conversions / clicks × 100
2,ROAS,revenue / ad_spend
3,profit,revenue - ad_spend
4,CPC,ad_spend / clicks
5,CPA,ad_spend / conversions


## Step 2.11 — Derived and Leakage-Prone Variables

Several variables in the dataset are mathematically derived from other campaign metrics.

The following relationships were identified:

- CTR = Clicks / Impressions × 100
- Conversion Rate = Conversions / Clicks × 100
- ROAS = Revenue / Ad Spend
- Profit = Revenue − Ad Spend
- CPC = Ad Spend / Clicks
- CPA = Ad Spend / Conversions

These variables will be retained in the cleaned dataset because they are important campaign performance and business metrics.

However, they will not automatically be used as predictors for every machine learning model. Model-specific feature selection will be performed later to prevent target leakage and avoid giving models direct components of their target variables.

For example:

- Clicks and Impressions should not be used as direct predictors when predicting CTR.
- Conversions and Clicks should not be used as direct predictors when predicting Conversion Rate.
- Revenue and Ad Spend should not be used as direct predictors when predicting ROAS.
- Revenue and Ad Spend should not be used as direct predictors when predicting Profit.

This approach preserves the information required for campaign analysis while ensuring that predictive models represent meaningful forecasting scenarios.

In [37]:
processed_path = "../data/processed/campaign_data_cleaned.csv"

df_clean.to_csv(
    processed_path,
    index=False
)

print("Cleaned dataset saved successfully.")
print("Location:", processed_path)

Cleaned dataset saved successfully.
Location: ../data/processed/campaign_data_cleaned.csv


In [38]:
preprocessing_summary = pd.DataFrame({
    "Check": [
        "Original Rows",
        "Processed Rows",
        "Original Columns",
        "Processed Columns",
        "Missing Values",
        "Duplicate Rows",
        "Invalid Dates",
        "Zero-Conversion Campaigns"
    ],
    "Result": [
        len(df),
        len(df_clean),
        df.shape[1],
        df_clean.shape[1],
        df_clean.isna().sum().sum(),
        df_clean.duplicated().sum(),
        df_clean["start_date"].isna().sum(),
        (df_clean["conversions"] == 0).sum()
    ]
})

preprocessing_summary

,Check,Result
0,Original Rows,10000
1,Processed Rows,10000
2,Original Columns,41
3,Processed Columns,40
4,Missing Values,0
5,Duplicate Rows,0
6,Invalid Dates,0
7,Zero-Conversion Campaigns,594


## Phase 2 — Current Preprocessing Status

The raw advertising campaign dataset has been loaded and a separate working copy has been created to preserve the original data.

The `start_date` variable was successfully converted from its original `DD-MM-YYYY` string format to a datetime representation. The initial generic conversion produced invalid values because the date format was not explicitly specified; after using the correct `%d-%m-%Y` format, all 10,000 dates were successfully converted with zero missing dates.

The dataset contains 594 campaigns with zero conversions (5.94%). These campaigns are valid observations and have been retained. Their stored CPA value of 0 has also been retained rather than introducing artificial missing or infinite values.

Categorical variables have been classified into nominal and ordinal groups. Encoding will be performed later using model pipelines rather than modifying the general cleaned dataset directly.

Derived and leakage-prone KPI relationships have been identified. The corresponding variables remain in the cleaned dataset because they are required for campaign performance analysis, but model-specific feature selection will later prevent target leakage.

The cleaned dataset has been saved as:

`data/processed/campaign_data_cleaned.csv`